# Seurat v5 scRNA-seq Workflow

> **ImmunoClassifier v0.1.0**

This notebook demonstrates a complete single-cell RNA-seq analysis workflow using:
- **R / Seurat v5** (via  %%R magic) — the canonical Seurat pipeline
- **Python / scanpy** — the Python-native equivalent used by ImmunoClassifier

## Setup

Install R and the Seurat package before running the R cells:



> **Note:** The R package name is  with a capital **S**.
> Using  (lowercase) will raise
> .


In [ ]:
# Enable rpy2 R magic
%load_ext rpy2.ipython

## 1. R / Seurat v5 Workflow

Run the full Seurat v5 analysis pipeline on a 10x CellRanger output.

**Key fix:** Use  with a capital **S** — R package names are case-sensitive.


In [ ]:
%%R
# ── R: Full Seurat v5 workflow ──────────────────────────────────────────────
# NOTE: Use library(Seurat) with a capital S — R package names are case-sensitive.
# Using library(seurat) (lowercase) raises:
#   Error in library(seurat) : there is no package called 'seurat'
library(Seurat)   # capital S
library(ggplot2)
library(dplyr)

# 1. Load 10x CellRanger output
counts <- Read10X(data.dir = "filtered_feature_bc_matrix/")
obj <- CreateSeuratObject(counts, project = "melanoma_TIL",
                          min.cells = 3, min.features = 200)

# 2. QC metrics
obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
obj[["percent.ribo"]] <- PercentageFeatureSet(obj, pattern = "^RP[SL]")

# Visualize QC
VlnPlot(obj, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3)

# 3. Filter
obj <- subset(obj,
    nFeature_RNA > 200 & nFeature_RNA < 5000 &
    nCount_RNA > 500 &
    percent.mt < 15   # tumor samples can have higher mito — adjust per dataset
)

# 4. Normalize + HVGs
obj <- NormalizeData(obj, normalization.method = "LogNormalize", scale.factor = 10000)
obj <- FindVariableFeatures(obj, selection.method = "vst", nfeatures = 2000)

# 5. Scale (regress out mito if it's confounding)
obj <- ScaleData(obj, vars.to.regress = "percent.mt")

# 6. PCA
obj <- RunPCA(obj, npcs = 50)
ElbowPlot(obj, ndims = 50)  # pick elbow — typically 20-30 PCs

# 7. Clustering
obj <- FindNeighbors(obj, dims = 1:30)
obj <- FindClusters(obj, resolution = 0.8)  # may need 0.4-1.2 range

# 8. UMAP
obj <- RunUMAP(obj, dims = 1:30)
DimPlot(obj, reduction = "umap", label = TRUE, repel = TRUE) + NoLegend()


## 2. Python / scanpy Equivalent

The same workflow implemented in Python using **scanpy** — the approach used by ImmunoClassifier.
No R installation required.


In [ ]:
import scanpy as sc
import numpy as np

# 1. Load 10x CellRanger output
adata = sc.read_10x_mtx(
    "filtered_feature_bc_matrix/",
    var_names="gene_symbols",
    cache=True,
)
adata.var_names_make_unique()

# 2. QC metrics
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.match(r"^RP[SL]")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo"], inplace=True)

# Visualize QC
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

# 3. Filter
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[
    (adata.obs.n_genes_by_counts < 5000)
    & (adata.obs.total_counts > 500)
    & (adata.obs.pct_counts_mt < 15)
].copy()

# 4. Normalize + HVGs
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts")

# 5. Scale (regress out mito)
adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.regress_out(adata_hvg, ["pct_counts_mt"])
sc.pp.scale(adata_hvg, max_value=10)

# 6. PCA
sc.tl.pca(adata_hvg, n_comps=50)
sc.pl.pca_variance_ratio(adata_hvg, n_pcs=50, log=True)  # equivalent to ElbowPlot

# 7. Clustering
sc.pp.neighbors(adata_hvg, n_pcs=30)
sc.tl.leiden(adata_hvg, resolution=0.8)

# 8. UMAP
sc.tl.umap(adata_hvg)
sc.pl.umap(adata_hvg, color="leiden", legend_loc="on data", title="Leiden Clusters")


## 3. ImmunoClassifier Integration

After preprocessing, feed the AnnData object directly into ImmunoClassifier:


In [ ]:
from immunoclassifier.data.preprocessing import preprocess
from immunoclassifier.models.xgboost_model import XGBoostClassifier

# Run the full ImmunoClassifier preprocessing pipeline
# (Replace adata with your loaded AnnData object from step 1)
# adata = preprocess(adata, min_genes=200, min_cells=3, max_pct_mito=15)

# Train XGBoost classifier on annotated reference data
# model = XGBoostClassifier(n_estimators=100, max_depth=6)
# model.train(adata_reference, label_key="cell_type")

# Predict on your query dataset
# predictions = model.predict(adata_query)
print("See examples/pbmc_classification_benchmark.ipynb for a full end-to-end example.")
